# 01_heart_preprocess

Selected source cells for review and reconstruction.

Source: `Heart_combined_samples/Heart_Combination_2(Sketch harmony).ipynb`

Raw Space Ranger outputs; creates on-disk matrices and Seurat objects.

Original execution counts and plots were removed. Do not run in the source-data folder: original output filenames and Docker paths are retained as provenance. Use the common scripts for new runs. These excerpts have not been executed as a fresh notebook.


Source cell 1 (one-based); original execution count 1.

In [ ]:
suppressPackageStartupMessages({
  library(Seurat)
  library(BPCells)
  library(Matrix)
  library(hdf5r)
})

Source cell 2 (one-based); original execution count 2.

In [ ]:
library(harmony)

Source cell 5 (one-based); original execution count 5.

In [ ]:
# 1) 4サンプルの outs を指定（ここが基準）
outs <- list(
  Heart_Aged_1  = "/work/Visium/Visium_session1/Heart_Aged_1/outs",
  Heart_Young_1 = "/work/Visium/Visium_session1/Heart_Young_1/outs",
  Heart_Aged_2  = "/work/Visium/Visium_session2/Heart_Aged_2/outs",
  Heart_Young_2 = "/work/Visium/Visium_session2/Heart_Young_2/outs"
)
stopifnot(all(dir.exists(unlist(outs))))

Source cell 6 (one-based); original execution count 6.

In [ ]:
# 2) 8um binned の h5 を直接指す（= square_008um/filtered_feature_bc_matrix.h5）
h5 <- lapply(outs, function(o) file.path(o, "binned_outputs/square_008um/filtered_feature_bc_matrix.h5"))
stopifnot(all(file.exists(unlist(h5))))

Source cell 7 (one-based); original execution count 7.

In [ ]:
# 3) on-disk matrix を作って保存するフォルダ（あなたの作業フォルダでOK）
odir <- "/work/Visium/Heart_combined_samples/on_disk_mats"
dir.create(odir, recursive = TRUE, showWarnings = FALSE)

Source cell 8 (one-based); original execution count 8.

In [ ]:
# 4) on-disk matrices 作成（10xガイドの “Create On-Disk Matrices” 相当）
mat_list <- lapply(names(h5), function(nm) {
  m <- BPCells::open_matrix_10x_hdf5(path = h5[[nm]], feature_type = "Gene Expression")
  out <- file.path(odir, nm)
  BPCells::write_matrix_dir(mat = m, dir = out, overwrite = TRUE)
  BPCells::open_matrix_dir(out)
})
names(mat_list) <- names(h5)

Source cell 9 (one-based); original execution count 9.

In [ ]:
# 5) Seurat object（on-diskのまま）
obj_list <- lapply(names(mat_list), function(nm) {
  so <- CreateSeuratObject(counts = mat_list[[nm]], project = nm, assay = "RNA",
                           min.cells = 0, min.features = 0)
  so$sample <- nm
  so <- RenameCells(so, add.cell.id = nm)
  so
})
names(obj_list) <- names(mat_list)

Source cell 10 (one-based); original execution count 10.

In [ ]:
# 6) merge（1つのSeuratに）
merge.obj <- Reduce(function(a,b) merge(a,b), obj_list)
merge.obj

Source cell 14 (one-based); original execution count 13.

In [ ]:
merge.obj <- subset(merge.obj, subset = nCount_RNA >= 25)

Source cell 15 (one-based); original execution count 14.

In [ ]:
# (2) Normalize / HVG（＝sketch前）
DefaultAssay(merge.obj) <- "RNA"
merge.obj <- NormalizeData(merge.obj, scale.factor = median(merge.obj$nCount_RNA), verbose = FALSE)
merge.obj <- FindVariableFeatures(merge.obj, verbose = FALSE)


Source cell 16 (one-based); original execution count 15.

In [ ]:
merge.obj <- SketchData(
  object = merge.obj,
  ncells = 50000,
  method = "LeverageScore",
  sketched.assay = "sketch"
)

Source cell 18 (one-based); original execution count 17.

In [ ]:
# (4) 以降は sketch
DefaultAssay(merge.obj) <- "sketch"

merge.obj <- NormalizeData(merge.obj, verbose = FALSE)
merge.obj <- FindVariableFeatures(merge.obj, verbose = FALSE)
merge.obj <- ScaleData(merge.obj, verbose = FALSE)
merge.obj <- RunPCA(merge.obj, npcs = 50, verbose = FALSE)


Source cell 19 (one-based); original execution count 18.

In [ ]:
# 7) Batch correction by Harmony（RunHarmonyではなく IntegrateLayers）
merge.obj <- IntegrateLayers(
  merge.obj,
  method = HarmonyIntegration,
  orig = "pca",
  new.reduction = "harmony",
  dims = 1:30
)

Source cell 20 (one-based); original execution count 19.

In [ ]:
# UMAP + clustering（harmony次元で）
merge.obj <- RunUMAP(merge.obj, reduction = "harmony", dims = 1:30, reduction.name = "umap")
merge.obj <- FindNeighbors(merge.obj, reduction = "harmony", dims = 1:30)
merge.obj <- FindClusters(merge.obj, resolution = 0.5)

Source cell 24 (one-based); original execution count 23.

In [ ]:
saveRDS(
  merge.obj,
  file = "heart_mergeobj_preProject_sketchHarmony.rds"
)

Source cell 26 (one-based); original execution count 25.

In [ ]:
merge.obj <- ProjectIntegration(
  object = merge.obj,
  sketched.assay = "sketch",
  assay = "RNA",
  reduction = "harmony"
)

Source cell 28 (one-based); original execution count 27.

In [ ]:
merge.obj <- ProjectData(
  object = merge.obj,
  assay = "RNA",
  full.reduction = "harmony.full",
  sketched.assay = "sketch",
  sketched.reduction = "harmony.full",
  umap.model = "umap",
  dims = 1:30,
  refdata = list(cluster_full = "seurat_clusters")
)

Source cell 30 (one-based); original execution count 29.

In [ ]:
saveRDS(
  merge.obj,
  file = "heart_mergeobj_preProject_full_sketchHarmony.rds"
)